In [ ]:
!pip install sentence-transformers openpyxl nltk -q

Step 1: Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/ShopHallu/01.ShopHallu_dataset_generated_for_rule_based_analysis_with_generation_text.xlsx"

df = pd.read_excel(
    file_path
)

print(df.shape)
df.head()

(22275, 14)


,asin,category,generation_reviews,generation_text,prompt_type,model,temperature,influencer_review,evidence_review,supported,fabricated_feature,exaggerated_performance,contradiction,emotional_only
0,B007JZU8N8,Accessories,"[""good to have, fits very well"", ""Expecting an...","good to have, fits very well Expecting anti-gl...",strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Screen protector is good but my was dead ..so..,NaN,NaN,NaN,NaN,NaN
1,NaN,Accessories,"[""good to have, fits very well"", ""Expecting an...","good to have, fits very well Expecting anti-gl...",strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Provides required protection without observing...,NaN,NaN,NaN,NaN,NaN
2,NaN,Accessories,"[""good to have, fits very well"", ""Expecting an...","good to have, fits very well Expecting anti-gl...",strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,ok,NaN,NaN,NaN,NaN,NaN
3,NaN,Accessories,"[""good to have, fits very well"", ""Expecting an...","good to have, fits very well Expecting anti-gl...",strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,I was pleasantly surprised at the thickness of...,NaN,NaN,NaN,NaN,NaN
4,NaN,Accessories,"[""good to have, fits very well"", ""Expecting an...","good to have, fits very well Expecting anti-gl...",strict,openai/gpt-4o-mini,0.0,Hey everyone! I wanted to share my thoughts on...,Ok,NaN,NaN,NaN,NaN,NaN


Step 2: Imports

In [ ]:
import re
import nltk

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Step 3: Load Sentence-BERT

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 4: Helper Functions

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        " ",
        text
    )

    return text

Keyword Overlap

In [ ]:
def get_keywords(text):

    text = clean_text(text)

    words = text.split()

    words = [
        w
        for w in words
        if w not in stop_words
        and len(w) > 2
    ]

    return set(words)

Similarity

In [ ]:
def sentence_similarity(
    evidence,
    generated
):

    emb1 = model.encode(
        str(evidence),
        convert_to_numpy=True
    )

    emb2 = model.encode(
        str(generated),
        convert_to_numpy=True
    )

    score = cosine_similarity(
        [emb1],
        [emb2]
    )[0][0]

    return round(float(score), 4)

Unsupported Keywords

In [ ]:
def unsupported_keywords(
    evidence,
    generated
):

    evidence_words = get_keywords(
        evidence
    )

    generated_words = get_keywords(
        generated
    )

    unsupported = generated_words - evidence_words

    return ", ".join(
        sorted(list(unsupported))[:20]
    )

Keyword Overlap Score

In [ ]:
def keyword_overlap(
    evidence,
    generated
):

    e = get_keywords(evidence)

    g = get_keywords(generated)

    overlap = e.intersection(g)

    count = len(overlap)

    ratio = (
        count / len(g)
        if len(g) > 0
        else 0
    )

    return count, round(ratio,4)

Exaggeration Detector

In [ ]:
exaggeration_words = [

    "perfect",
    "best",
    "amazing",
    "life changing",
    "must have",
    "flawless",
    "completely",
    "instantly",
    "guaranteed",
    "ultimate",
    "transform",
    "transformative",
    "cure",
    "overnight",
    "everyone",
    "always",
    "never",
    "incredible"
]

In [ ]:
def exaggeration_flag(text):

    text = str(text).lower()

    found = []

    for word in exaggeration_words:

        if word in text:
            found.append(word)

    return ", ".join(found)

Weak Suggested Label

In [ ]:
def weak_label(
    similarity,
    overlap_ratio,
    exaggeration
):

    if similarity < 0.30:
        return "Review_Needed"

    if exaggeration != "":
        return "Potential_Hallucination"

    if similarity > 0.75:
        return "Likely_Supported"

    return "Review_Needed"

Step 5: Run Analysis

In [ ]:
similarity_scores = []
overlap_counts = []
overlap_ratios = []
unsupported_list = []
exaggeration_list = []
suggestions = []

for idx,row in df.iterrows():

    evidence = row["evidence_review"]
    generated = row["influencer_review"]

    sim = sentence_similarity(
        evidence,
        generated
    )

    count,ratio = keyword_overlap(
        evidence,
        generated
    )

    unsupported = unsupported_keywords(
        evidence,
        generated
    )

    exaggeration = exaggeration_flag(
        generated
    )

    suggestion = weak_label(
        sim,
        ratio,
        exaggeration
    )

    similarity_scores.append(sim)

    overlap_counts.append(count)

    overlap_ratios.append(ratio)

    unsupported_list.append(
        unsupported
    )

    exaggeration_list.append(
        exaggeration
    )

    suggestions.append(
        suggestion
    )

Step 6: Add New Columns

In [ ]:
df["similarity_score"] = similarity_scores

df["keyword_overlap_count"] = overlap_counts

df["keyword_overlap_ratio"] = overlap_ratios

df["unsupported_keywords"] = unsupported_list

df["exaggeration_flag"] = exaggeration_list

df["weak_annotation_signal"] = suggestions

df["annotator_comment"] = ""

Step 7: Save Annotation Template

In [ ]:
output_file = (
    "02.Rule_Based_Analysis_for_Annotation.xlsx"
)

df.to_excel(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

Saved: 02.Rule_Based_Analysis_for_Annotation.xlsx


In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

# Save dataframe as Excel
output_file = "02.Rule_Based_Analysis_for_Annotation.xlsx"

df.to_excel(
    output_file,
    index=False
)

# Load workbook
wb = load_workbook(output_file)
ws = wb.active

# -------------------------
# Column Width Settings
# -------------------------

column_widths = {

    "A": 18,   # sample_id / asin
    "B": 15,   # category
    "C": 60,   # generation_reviews
    "D": 60,   # generation_text
    "E": 15,   # prompt_type
    "F": 18,   # model
    "G": 12,   # temperature

    "H": 100,  # paragraph
    "I": 100,  # evidence_review

    "J": 12,   # supported

    "K": 18,   # Fabricated_Feature
    "L": 22,   # Exaggerated_Performance
    "M": 18,   # Contradiction
    "N": 18,   # emotional_only

    "O": 15,   # similarity_score
    "P": 18,   # keyword_overlap_count
    "Q": 18,   # keyword_overlap_ratio

    "R": 50,   # unsupported_keywords
    "S": 40,   # exaggeration_flag
    "T": 25,   # weak_annotation_signal

    "U": 70    # annotator_comment
}

for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

# -------------------------
# Header Formatting
# -------------------------

for cell in ws[1]:
    cell.font = Font(bold=True)

# -------------------------
# Wrap Text
# -------------------------

for row in ws.iter_rows():

    for cell in row:

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )

# -------------------------
# Freeze Header Row
# -------------------------

ws.freeze_panes = "A2"

# -------------------------
# Add Filter
# -------------------------

ws.auto_filter.ref = ws.dimensions

# -------------------------
# Optional Row Height
# -------------------------

for row in range(2, ws.max_row + 1):
    ws.row_dimensions[row].height = 80

# Save
wb.save(output_file)

print(f"Saved: {output_file}")


Saved: 02.Rule_Based_Analysis_for_Annotation.xlsx


Bonus Statistics

In [ ]:
print(
    df["suggested_label"]
    .value_counts()
)

suggested_label
Potential_Hallucination    13560
Review_Needed               8469
Likely_Supported             246
Name: count, dtype: int64


In [ ]:
print(
    df["similarity_score"]
    .describe()
)

count    22275.000000
mean         0.448028
std          0.200629
min         -0.103200
25%          0.296900
50%          0.473300
75%          0.605500
max          0.881200
Name: similarity_score, dtype: float64
